# Session 4: register, deploy and chat with Phi-3
Use the Python 3.11 `.venv-session4` kernel after installing the `app` and `evaluation` extras. This notebook uses the adapters from the successful Session 3 run. Cloud registration/deployment cells create real resources. Finish with the cleanup cell after testing the app, as required by the practice.

In [ ]:
from pathlib import Path
import importlib, json, os, subprocess, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
from agentic_workflows.handler import EndpointHandler
from agentic_workflows.inference import download_model
from agentic_workflows.online_prediction import endpoint_url, post_prediction, prediction_text
source = json.loads((ROOT / 'data/session3/run_summary.json').read_text())
model_uri = source['artifacts']['fine-tuning-component/model']['uri']
print('Source model:', model_uri)
print('Source pipeline:', source['job'])


## 1. Inspect the request and test the handler locally
The standalone handler loads the base Phi-3 model and attaches the saved LoRA adapters. Its request has `instances` and `parameters`; its response has `predictions`. A full local model load needs the training dependencies and sufficient RAM, preferably an NVIDIA GPU. The unit tests exercise this contract on CPU using lightweight test doubles, and the deployed endpoint verifies real model loading/generation.

In [ ]:
request = json.loads((ROOT / 'examples/prediction_request.json').read_text())
display(request, EndpointHandler.validate_request(request))
assert request['instances'][0]['prompt'].endswith('<|assistant|>' + chr(10))


In [ ]:
# Optional local GPU experiment: install the training extra before running this cell.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Use the deployed endpoint below; local Phi-3 loading is optional on this CPU machine.')
local_dir = ROOT / 'models/phi3-yoda-adapter'
if not (local_dir / 'adapter_config.json').exists():
    download_model(model_uri, str(local_dir))
handler = EndpointHandler(str(local_dir))
result = handler(request)
assert 'predictions' in result
display(result)


## 2. Upload the handler and register the first model version
The script uploads `handler.py` and the serving `requirements.txt` into the model artifact directory, then calls `aiplatform.Model.upload` with the course Hugging Face container and port 8080. It records source-pipeline provenance and SHA-256 hashes in `data/session4/registration.json`. Registered resources can be inspected in Agent Platform → Models → Model registry, in `europe-west2`.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts/register_model_with_custom_handler.py'), '--model-uri', model_uri, '--model-name', 'lulik-yoda-phi3', '--version-alias', 'session4-initial'], cwd=ROOT, check=True)
registration = json.loads((ROOT / 'data/session4/registration.json').read_text())
display(registration['registrations'][-1])


## 3. Register a second version under the same model
This demonstrates Model Registry versioning using `parent_model`. The weights are intentionally the same; a new registration does not retrain the model.

In [ ]:
parent = registration['registrations'][-1]['model_resource']
subprocess.run([sys.executable, str(ROOT / 'scripts/register_model_with_custom_handler.py'), '--model-uri', model_uri, '--parent-model', parent, '--version-alias', 'session4-versioned'], cwd=ROOT, check=True)
registration = json.loads((ROOT / 'data/session4/registration.json').read_text())
display(registration)


## 4. Deploy to a temporary endpoint
The script provisions `n1-standard-8`, one T4 GPU, and exactly one replica. Deployment can take 15–30 minutes. It saves the endpoint manifest and updates the ignored `.env` file when ready. You can create the same deployment from the Model Registry console.

In [ ]:
model_resource = registration['registrations'][-1]['versioned_model_resource']
subprocess.run([sys.executable, str(ROOT / 'scripts/manage_endpoint.py'), 'deploy', '--model', model_resource], cwd=ROOT, check=True)
load_dotenv(ROOT / '.env', override=True)
import agentic_workflows.constants as constants
importlib.reload(constants)
display(json.loads((ROOT / 'data/session4/endpoint.json').read_text()))


## 5. Make an authenticated HTTP prediction
`AUTH_MODE=adc` uses Application Default Credentials and refreshes access tokens automatically; this works with the existing course credentials even when the `gcloud` CLI is not installed. `AUTH_MODE=gcloud` uses the practice's `subprocess.check_output(['gcloud', 'auth', 'print-access-token'])` method. Tokens stay in memory and are never saved or printed.

In [ ]:
url = endpoint_url(constants.GCP_PROJECT_NUMBER or constants.GCP_PROJECT_ID, constants.GCP_REGION, constants.GCP_ENDPOINT_ID)
response = post_prediction(url, request, auth_mode=constants.AUTH_MODE)
display(response)
print(prediction_text(response))


## 6. Chat in Chainlit
From a terminal in the project directory, run:
```powershell
.\.venv-session4\Scripts\python.exe scripts/run_chat.py
```
Open http://localhost:8000 and send an English sentence. The app applies the saved tokenizer's chat template, calls the protected Vertex endpoint through HTTP, and displays only the generated answer. Restart the app after changing endpoint settings in `.env`.

## 7. Cleanup after the practice
This undeploys the model and deletes only the endpoint recorded in the Session 4 manifest, after checking its ownership labels. The registered model versions and source adapters remain available. `.env` is updated to clear the removed endpoint ID. Redeploy and restart Chainlit when you want another live chat session.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts/manage_endpoint.py'), 'cleanup'], cwd=ROOT, check=True)
display(json.loads((ROOT / 'data/session4/endpoint.json').read_text()))
